In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import gurobipy as gp
from gurobipy import GRB

# Data initialization

In [ ]:
# initialize relevant data
current_filepath = Path('.').resolve().parent
data_path = current_filepath / 'raw-data'

branches_df = pd.read_csv(filepath_or_buffer=data_path / 'branches.csv', skiprows=1)
buses_df = pd.read_csv(filepath_or_buffer=data_path / 'buses.csv', skiprows=1)
ptdf_df = pd.read_csv(filepath_or_buffer=data_path / 'ptdf.csv', skiprows=1)
sf_contingencies_df = pd.read_csv(filepath_or_buffer=data_path / 'sf-contingencies.csv', skiprows=1)

# The generator schedule was produced by Gemini using the following prompt:
"""
Using the load information in @file:buses.csv provide a .csv with the following columns: 

> Gen ID,Bus Number,Cost,Pmin,Pmax

Following these characeristics:
1. Generators are labeled G_i_k starting at i=0,k=0 where i refers to the bus it is connected to, k enumerates the k'th generator on the node, which sometimes happens if there are more than 1 generator on each node
2. Bus number is the bus it is connected to
3. Cost is the $ per MWh demanded by the generator
4. Pmin is the min amount of power that the generator is willing to sell, if at all
5. Pmax is the maximum amount of power it can supply

Additionally, to make the problem interesting, formulate the problem so that you can ensure, or set up so that at a high probability:
1. Some nodes have more than 1 generator.
2. For some nodes where there are more than 1 generator, make it so that their costs are different, but both are still cleared. 
3. There will be congestion in the network. That is, cheaper power flow is possible, but there are thermal constraints. [For this requirement, do not perform any analysis, simply set up the numbers so that there will be a high proability of congestion].
"""
# AI Rationale: In a real-world setting, this data would be accessed through historical archives
# However our group does not have access to such archives from the US Power Grid Operators
# Therefore we determine this to be out of scope, and so we fall back to using an AI-provided dataset
generators_df = pd.read_csv(filepath_or_buffer=data_path / 'generators.csv')

In [3]:
generator_min = generators_df['Pmin'].to_numpy()
generator_max = generators_df['Pmax'].to_numpy()
generator_cost = generators_df['Cost'].to_numpy()
generator_node = generators_df['Bus Number'].to_numpy()

nominal_demand = buses_df['Load MW'].replace([np.nan], 0).to_numpy()
load_node = buses_df['Number'].to_numpy()
load_node = load_node[nominal_demand != 0]
nominal_demand = nominal_demand[nominal_demand != 0]

G = generator_cost.shape[0]
D = nominal_demand.shape[0]

thermal_flow_lim = branches_df['Lim MVA A'].to_numpy()

power_balance_cons_vec = np.concatenate((np.ones((G, 1)), -1*np.ones((D, 1)))).T

## Injection Mapping

In [4]:
gen_node_map = np.zeros((buses_df.shape[0], G))
gen_node_map[generator_node - 1, np.arange(G)] = 1
load_node_map = np.zeros((buses_df.shape[0], load_node.shape[0]))
load_node_map[load_node - 1, np.arange(load_node.shape[0])] = -1

injection_map = np.hstack((gen_node_map, load_node_map))

## Shift Factor to Thermal Flow Mapping

In [5]:
# Create a mapping from line to thermal flow lim
thermal_flow_mapping = branches_df[['From Name', 'To Number', 'Lim MVA A']]
sf_no_contingency = ptdf_df.loc[:, '1  TO  2 CKT 1':'21  TO  22 CKT 1']

# match the thermal flow lim to the shift factor values
sf_names = sf_no_contingency.columns.values
thermal_flow_lim = np.zeros((sf_names.shape[0]))
for (ind, line) in enumerate(sf_names):
    line_string = line.split(' ')
    from_no = int(line_string[0])
    to_no = int(line_string[4])
    thermal_flow_lim[ind] = thermal_flow_mapping.query('`From Name` == @from_no and `To Number` == @to_no')['Lim MVA A'].iloc[0]

sf_no_contingency = sf_no_contingency.to_numpy().T

In [6]:
sf_contingency = sf_contingencies_df.loc[:, '9 (MONITOR_1_1-2_CONTINGENCY_10_8-9)':'304 (MONITOR_9_7-8_CONTINGENCY_8_5-10)']

# match the thermal flow lim to the shift factor contingency values
contingency_names = sf_contingency.columns.values
thermal_flow_lim_contingencies = np.zeros((contingency_names.shape[0]))
for (ind, contingency) in enumerate(contingency_names):
    line_string = contingency.split('_')[2]
    from_no = int(line_string.split('-')[0])
    to_no = int(line_string.split('-')[1])
    thermal_flow_lim_contingencies[ind] = thermal_flow_mapping.query('`From Name` == @from_no and `To Number` == @to_no')['Lim MVA A'].iloc[0]

sf_contingency = sf_contingency.to_numpy().T

In [7]:
base_case_trans_cons_mat = sf_no_contingency @ injection_map
contingency_trans_cons_mat = sf_contingency @ injection_map

## Model Set-up

In [8]:
lp_model = gp.Model()
generators = lp_model.addMVar(G, name='generators')
lp_model.setMObjective(Q=None, c=generator_cost, constant=0, sense=GRB.MINIMIZE)
loads = lp_model.addMVar(D, name='load_served', lb=0.0)
power_balance_cons = lp_model.addMConstr(power_balance_cons_vec, gp.hstack((generators, loads)), '=', [0], 'power balance')
generator_min_cons = lp_model.addMConstr(np.eye(G), generators, '>=', generator_min, 'min generator capacity')
generator_max_cons = lp_model.addMConstr(np.eye(G), generators, '<=', generator_max, 'max generator capacity')
load_limits_cons = lp_model.addMConstr(np.eye(D), loads, '<=', nominal_demand, 'max load demand')
base_case_cons_min = lp_model.addMConstr(base_case_trans_cons_mat, gp.hstack((generators, loads)), '>=', -1*thermal_flow_lim, 'Base Case Transimission Min')
base_case_cons_max = lp_model.addMConstr(base_case_trans_cons_mat, gp.hstack((generators, loads)), '<=', thermal_flow_lim, 'Base Case Transimission Max')
contingency_cons_min = lp_model.addMConstr(contingency_trans_cons_mat, gp.hstack((generators, loads)), '>=', -1*thermal_flow_lim_contingencies, 'Contingency Transimission Min')
contingency_cons_max = lp_model.addMConstr(contingency_trans_cons_mat, gp.hstack((generators, loads)), '<=', thermal_flow_lim_contingencies, 'Contingency Transimission Max')

lp_model.update()
lp_model.write('./DA_Clearance.lp')
lp_model.optimize()

Set parameter Username
Set parameter LicenseID to value 2762801
Academic license - for non-commercial use only - expires 2027-01-10
Set parameter LicenseID to value 2762801
Academic license - for non-commercial use only - expires 2027-01-10
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Max
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 2959 rows, 43 columns and 112859 nonzeros (Min)
Model fingerprint: 0x1a38f847
Model has 27 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-04, 1e+00]
  Objective range  [1e+01, 6e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 5e+02]

Presolve removed 2388 rows and 0 columns
Presolve time: 0.01s
Presolved: 571 rows, 306 columns, 23918 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4900000e+03   1.375000e+01   0.000000e+00      0s
       2    2.4900000e+03

## Solutions

In [9]:
power_produced = generators.X
power_served = loads.X

variable_ids = np.hstack([np.char.add('Gen_', generator_node.astype(str)), np.char.add('Load_', load_node.astype(str))])
bus_number = np.hstack([generator_node, load_node])
variable_types = np.hstack([np.full(generator_node.shape[0], 'Generator'), np.full(load_node.shape[0], 'Load')]) 
values = np.hstack([power_produced, power_served])

solutions_df = pd.DataFrame()

solutions_df['Variable ID'] = variable_ids
solutions_df['Bus Number'] = bus_number
solutions_df['Type'] = variable_types
solutions_df['Value'] = values

solutions_df.to_csv('./solutions.csv')

In [10]:
solutions_df["Type"].value_counts()

Type
Generator    27
Load         16
Name: count, dtype: int64

# Solution Analysis

In [11]:
gens_solns = solutions_df[solutions_df['Type'] == 'Generator']
gens_solns.head(2)

,Variable ID,Bus Number,Type,Value
0,Gen_1,1,Generator,0.0
1,Gen_2,2,Generator,0.0


In [12]:
load_solns = solutions_df[solutions_df['Type'] == 'Load']
load_solns.head(2)

,Variable ID,Bus Number,Type,Value
27,Load_2,2,Load,97.0
28,Load_3,3,Load,0.0


In [13]:
lp_model.printAttr('Pi', "*")


  Constraint           Pi 
-------------------------
min generator capacity[8]           33 
min generator capacity[11]           38 
min generator capacity[13]           34 
min generator capacity[15]           40 
min generator capacity[16]           35 
min generator capacity[20]           11 
min generator capacity[22]           12 
min generator capacity[24]           13 


In [19]:
dispatch_vector = np.hstack((generators.X, loads.X))
base_case_flow = base_case_trans_cons_mat @ dispatch_vector

line_loadings_df = pd.DataFrame({
    'Line': sf_names,
    'From Bus': [int(name.split()[0]) for name in sf_names],
    'To Bus': [int(name.split()[2]) for name in sf_names],
    'Flow MW': base_case_flow,
    'Limit MW': thermal_flow_lim
})

line_loadings_df['Abs Flow MW'] = np.abs(line_loadings_df['Flow MW'])
line_loadings_df['% Max Load'] = 100 * line_loadings_df['Abs Flow MW'] / line_loadings_df['Limit MW']
line_loadings_df = line_loadings_df.sort_values('% Max Load', ascending=False).reset_index(drop=True)

line_loadings_df.sort_values("% Max Load", ascending=False).head(20)

,Line,From Bus,To Bus,Flow MW,Limit MW,Abs Flow MW,% Max Load,Headroom MW
0,1 TO 2 CKT 1,1,2,49.6417,175.0,49.6417,28.366686,125.3583
1,5 TO 10 CKT 1,5,10,-33.7529,175.0,33.7529,19.287371,141.2471
2,3 TO 1 CKT 1,3,1,28.8888,175.0,28.8888,16.507886,146.1112
3,2 TO 4 CKT 1,2,4,-26.0351,175.0,26.0351,14.877200,148.9649
4,4 TO 9 CKT 1,4,9,-26.0351,175.0,26.0351,14.877200,148.9649
5,2 TO 6 CKT 1,2,6,-21.3242,175.0,21.3242,12.185257,153.6758
6,10 TO 6 CKT 1,10,6,21.3242,175.0,21.3242,12.185257,153.6758
7,1 TO 5 CKT 1,1,5,-20.7529,175.0,20.7529,11.858800,154.2471
8,24 TO 3 CKT 1,24,3,29.2155,400.0,29.2155,7.303875,370.7845
9,11 TO 10 CKT 1,11,10,26.7970,400.0,26.7970,6.699250,373.2030


In [22]:
# Duals
base_case_cons_max.Pi

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0.])

In [23]:
base_case_cons_min.Pi

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0.])

## Problem with small model:
We are unable to get any congestion